# 3. 2D floor planning with rotation: cover as much floor area as possible

Lecture: *Week 11 - Floor Planning* (variant of the minimise-area model).

*Python version of the MATLAB file `5 Source Files/MATLAB/floor_planning_rotation_2d_max_item_area.m` (same data and constraints; big-M reduced from 9999 to a safe tight value, see the data cell), written for this repository. Runs in the browser with HiGHS.*

The six lecture items (total area 124) do not all fit on a 10 x 10 floor (area 100).
Each item may be left out (u = 0); the objective is the total area of the items placed.

*Several layouts can be optimal, so the positions drawn may differ between runs or solvers; the objective value is the same.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Data (as in the MATLAB file: the six items from the lecture)
w = np.array([4,3,6,7,3,2])
h = np.array([5,7,4,6,3,4])
W, H = 10, 10    # floor size
# Big-M: the MATLAB file uses M = 9999; a tight value is numerically safer with HiGHS (see notebook 2).
M = W + H + max(w.max(), h.max())
no_items = len(w)

In [ ]:
# Model
model = ConcreteModel()
model.I = RangeSet(no_items)
model.J = RangeSet(no_items)
model.w = Param(model.I, initialize=lambda model, i: int(w[i-1]))
model.h = Param(model.I, initialize=lambda model, i: int(h[i-1]))

model.x = Var(model.I, within=NonNegativeReals)      # bottom-left corner
model.y = Var(model.I, within=NonNegativeReals)
model.xij = Var(model.I, model.J, within=Binary)     # which side of each other two items are
model.yij = Var(model.I, model.J, within=Binary)
model.r = Var(model.I, within=Binary)                # 1 = item rotated 90 degrees
model.u = Var(model.I, within=Binary)                # 1 = item placed on the floor

In [ ]:
# Objective
model.obj = Objective(expr=sum(model.u[i] * int(w[i-1] * h[i-1]) for i in model.I), sense=maximize)

In [ ]:
# Constraints: two placed items must not overlap (switched off by M if either item is left out)
def width(model, i):   # width of item i after rotation
    return (1 - model.r[i]) * model.w[i] + model.r[i] * model.h[i]

def height(model, i):  # height of item i after rotation
    return (1 - model.r[i]) * model.h[i] + model.r[i] * model.w[i]

def overlap_xij_rule(model, i, j):
    if i < j:
        return model.x[i] + width(model, i) <= model.x[j] + M*(model.xij[i,j] + model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_xij = Constraint(model.I, model.J, rule=overlap_xij_rule)

def overlap_xji_rule(model, i, j):
    if i < j:
        return model.x[j] + width(model, j) <= model.x[i] + M*(1 - model.xij[i,j] + model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_xji = Constraint(model.I, model.J, rule=overlap_xji_rule)

def overlap_yij_rule(model, i, j):
    if i < j:
        return model.y[i] + height(model, i) <= model.y[j] + M*(1 + model.xij[i,j] - model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_yij = Constraint(model.I, model.J, rule=overlap_yij_rule)

def overlap_yji_rule(model, i, j):
    if i < j:
        return model.y[j] + height(model, j) <= model.y[i] + M*(2 - model.xij[i,j] - model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_yji = Constraint(model.I, model.J, rule=overlap_yji_rule)

# a placed item must lie inside the W x H floor
model.box_xi = Constraint(model.I, rule=lambda model, i: model.x[i] + width(model, i) <= W + M*(1 - model.u[i]))
model.box_yi = Constraint(model.I, rule=lambda model, i: model.y[i] + height(model, i) <= H + M*(1 - model.u[i]))

In [ ]:
# Solve (the MATLAB original stops after 10 s: intlinprog(problem, 'MaxTime', 10))
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)
print("Status:", results.solver.termination_condition)
print("Objective:", value(model.obj))

In [ ]:
# Draw the placed items (an item is placed / rotated when its binary is 1; test > 0.5, not truthiness)
placed = [i for i in model.I if model.u[i].value > 0.5]
print("Placed items:", placed, "of", list(model.I))
for i in placed:
    rot = model.r[i].value > 0.5
    print(f"item {i}: {w[i-1]}x{h[i-1]}{' rotated' if rot else ''} at ({model.x[i].value:g}, {model.y[i].value:g})")

fig, ax = plt.subplots()
cmap = plt.get_cmap('Pastel1')
for i in placed:
    wi, hi = (h[i-1], w[i-1]) if model.r[i].value > 0.5 else (w[i-1], h[i-1])
    ax.add_patch(patches.Rectangle((model.x[i].value, model.y[i].value), wi, hi, linewidth=1.5,
                                   edgecolor='black', facecolor=cmap(i / no_items), label=str(i)))
ax.add_patch(patches.Rectangle((0, 0), W, H, fill=False, linestyle='--', edgecolor='grey'))
ax.set_xlim(-1, W + 1); ax.set_ylim(-1, H + 1); ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.grid(True); ax.legend(loc='upper right')
plt.show()